In [0]:
%sql
CREATE CATALOG IF NOT EXISTS 3po;
CREATE SCHEMA IF NOT EXISTS 3po.main;
CREATE SCHEMA IF NOT EXISTS 3po.responses;
CREATE TABLE IF NOT EXISTS 3po.main.timeline (
time INTEGER
)
USING DELTA
TBLPROPERTIES (
  'delta.appendOnly' = 'true',
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

CREATE TABLE IF NOT EXISTS 3po.main.ingests (
time INTEGER,
instances INTEGER,
posts INTEGER,
files INTEGER
)
USING DELTA
TBLPROPERTIES ('delta.appendOnly' = 'true');

In [0]:
def contracts():
    # --- Check if volume exists ---
    existing_volumes = [
        row.volume_name
        for row in spark.sql("SHOW VOLUMES IN 3po.main").collect()
    ]

    if "contracts" in existing_volumes:
        print("Volume 3po.main.contracts already exists. Skipping creation.")
        return

    # --- Create the volume ---
    spark.sql("CREATE VOLUME 3po.main.contracts")
    print("Created volume 3po.main.contracts.")

    # --- Define YAML file contents ---
    yaml_files = {
        "history.yaml": """\
$schema: "https://json-schema.org/draft/2020-12/schema"
$id: "3po/contracts/history.yaml"
version: "1.0.0"
title: "history"
description: >
  Schema shared by all tables in any schema under catalog 3po
  whose name starts with 'h'.
applies_to:
  catalog: "3po"
  schema_pattern: "^h"
  type: table
properties:
  time:
    type: integer
    nullable: false
  impressions:
    type: integer
    nullable: false
    minimum: 0
required:
  - time
  - impressions
""",

        "ingests.yaml": """\
$schema: "https://json-schema.org/draft/2020-12/schema"
$id: "3po/contracts/ingests.yaml"
version: "1.0.0"
title: "ingests"
description: >
  Schema for the ingests table in 3PO.main.
applies_to:
  catalog: "3PO"
  schema: "main"
  table: "ingests"
  type: table
properties:
  time:
    type: integer
    nullable: false
  instances:
    type: integer
    nullable: false
    minimum: 0
  posts:
    type: integer
    nullable: false
    minimum: 0
  files:
    type: integer
    nullable: false
    minimum: 0
required:
  - time
  - instances
  - posts
  - files
""",

        "positivity.yaml": """\
$schema: "https://json-schema.org/draft/2020-12/schema"
$id: "3po/contracts/positivity.yaml"
version: "1.0.0"
title: "positivity"
description: >
  Schema shared by all tables in 3PO.main, excluding 'timeline' and 'ingests'.
applies_to:
  database: "3PO"
  schema: "main"
  type: table
  exclude_tables:
    - timeline
    - ingests
properties:
  id:
    type: string
    nullable: false
    minLength: 1
  positivity:
    type: ["number", "null"]
    format: float
    minimum: -1.0
    maximum: 1.0
required:
  - id
  - positivity
""",

        "timeline.yaml": """\
$schema: "https://json-schema.org/draft/2020-12/schema"
$id: "3po/contracts/timeline.yaml"
version: "1.0.0"
title: "timeline"
description: >
  Schema for the timeline table in 3PO.main.
applies_to:
  catalog: "3PO"
  schema: "main"
  table: "timeline"
  type: table
properties:
  time:
    type: integer
    nullable: false
additionalProperties:
  type: ["number", "null"]
  format: float
  minimum: -1.0
  maximum: 1.0
required:
  - time
""",

        "timestamp.yaml": """\
$schema: "https://json-schema.org/draft/2020-12/schema"
$id: "3po/contracts/timestamp.yaml"
version: "1.0.0"
title: "timestamp"
description: >
  Schema shared by all tables in any schema under catalog 3PO
  whose name starts with 't'.
applies_to:
  catalog: "3PO"
  schema_pattern: "^t"
  type: table
properties:
  id:
    type: string
    nullable: false
    minLength: 1
  impressions:
    type: integer
    nullable: false
    minimum: 0
  positivity:
    type: ["number", "null"]
    minimum: -1.0
    maximum: 1.0
required:
  - id
  - impressions
  - positivity
""",
    }

    # --- Write YAML files to the volume ---
    base_path = "/Volumes/3po/main/contracts"

    for filename, content in yaml_files.items():
        path = f"{base_path}/{filename}"
        dbutils.fs.put(path, content, overwrite=True)
        print(f"  Written: {path}")

    print("Volume setup complete.")


contracts()